In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_data.csv")

df.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


- Binary Encoding (Yes/No → 1/0)

In [2]:

binary_cols = [
    'partner', 'dependents', 'phoneservice', 'paperlessbilling', 'churn'
]

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

## Tenure Group Feature

In [3]:
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins = [0,12,24,48,72],
    labels=['0-1Y','1-2Y','2-4Y','4+Y']
)

## Monthly Charges Bucket

In [4]:
df['revenue_bucket'] = pd.cut(
    df['monthlycharges'],
    bins = [0,20,70,120],
    labels=['Low','Medium','High']
)

## Total Services Count

In [5]:
services = [
    'onlinesecurity','onlinebackup','deviceprotection',
    'techsupport','streamingtv','streamingmovies']
df['total_services']= df[services].apply(
    lambda x: (x == 'Yes').sum(), axis = 1)


## High Value Customer Flag

In [6]:
df['high_value_customer'] = (
    (df['monthlycharges'] > 70) & (df['tenure'] > 24)).astype(int)

## One-Hot Encoding (categorical)

In [7]:
df = pd.get_dummies(df, drop_first=True)

In [13]:
df.head()

,seniorcitizen,partner,dependents,tenure,phoneservice,paperlessbilling,monthlycharges,totalcharges,churn,total_services,...,contract_One year,contract_Two year,paymentmethod_Credit card (automatic),paymentmethod_Electronic check,paymentmethod_Mailed check,tenure_group_1-2Y,tenure_group_2-4Y,tenure_group_4+Y,revenue_bucket_Medium,revenue_bucket_High
0,0,1,0,1,0,1,29.85,29.85,0,1,...,False,False,False,True,False,False,False,False,True,False
1,0,0,0,34,1,0,56.95,1889.50,0,2,...,True,False,False,False,True,False,True,False,True,False
2,0,0,0,2,1,1,53.85,108.15,1,2,...,False,False,False,False,True,False,False,False,True,False
3,0,0,0,45,0,0,42.30,1840.75,0,3,...,True,False,False,False,False,False,True,False,True,False
4,0,0,0,2,1,1,70.70,151.65,1,0,...,False,False,False,True,False,False,False,False,False,True


In [14]:
df.shape

(7043, 7080)

In [15]:
df.to_csv("../data/processed/feature_engineered_data.csv", index=False)